# RealWorld HAR — Feature Extraction for Machine Learning

This notebook loads the segment-aware cleaned dataset (`realworld_clean.pkl`)
and produces a windowed feature table ready for classical ML models
(e.g. Random Forest).

**Key fix vs. the previous version:** windowing and gravity filtering are
now done *within each continuous segment* (`segment_id`), never across a
recording gap. This prevents filter artifacts and windows that mix two
unrelated recording sessions.

Pipeline:
1. Load cleaned data + verify `segment_id` is present
2. Gravity filter (per segment)
3. Sensor synchronization (acc + gyr + mag, per segment)
4. Sliding-window feature extraction (per segment)
5. Save the final feature table

In [1]:
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt, periodogram
from scipy.stats import skew, kurtosis

from pathlib import Path

pd.set_option("display.max_columns", 50)

In [2]:
DATA_DIR = Path("/lustre09/project/6081099/reem2005/DATASET/PROCESSED")
CLEAN_INPUT = DATA_DIR / "realworld_clean.pkl"
FEATURES_OUTPUT = DATA_DIR / "realworld_features.pkl"

final_df = pd.read_pickle(CLEAN_INPUT)

print("Shape:", final_df.shape)
print("Columns:", list(final_df.columns))

assert "segment_id" in final_df.columns, "segment_id missing — did you load the wrong file?"
print("\n✅ segment_id present. Unique segments:",
      final_df.groupby(["participant", "sensor", "position", "activity", "segment_id"], observed=True).ngroups)

Shape: (72309356, 10)
Columns: ['id', 'attr_time', 'attr_x', 'attr_y', 'attr_z', 'participant', 'activity', 'sensor', 'position', 'segment_id']

✅ segment_id present. Unique segments: 51813


In [3]:
FS = 50                # sampling rate (Hz), confirmed in the EDA notebook
GRAVITY_CUTOFF = 0.3    # Hz, low-pass cutoff to isolate gravity component
FILTER_ORDER = 4

WINDOW_SIZE = 100       # 2 seconds @ 50Hz
STEP_SIZE = 50          # 50% overlap

MERGE_TOLERANCE_MS = 10 # max time difference allowed when aligning sensors

SENSORS = ["acc", "gyr", "mag"]

In [4]:
def gravity_filter(signal: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Low-pass Butterworth filter to separate the gravity component
    from body acceleration. Must be called on a single continuous
    segment only — applying it across a time gap produces ringing
    artifacts at the gap boundary."""
    nyquist = FS / 2
    b, a = butter(FILTER_ORDER, GRAVITY_CUTOFF / nyquist, btype="low")
    gravity = filtfilt(b, a, signal)
    body = signal - gravity
    return body, gravity

Sliding Window Generator

In [5]:
def create_sliding_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE, step_size: int = STEP_SIZE):
    """Yields fixed-length windows by row position. Safe to use here
    ONLY because the caller guarantees `df` is a single continuous
    segment (no internal time gaps)."""
    for start in range(0, len(df) - window_size + 1, step_size):
        yield df.iloc[start:start + window_size]

In [6]:
def extract_features(window: pd.DataFrame) -> dict:
    """Extracts time-domain and frequency-domain features from one window."""
    features = {}
    sensor_columns = [
        "body_x", "body_y", "body_z",
        "gyr_x", "gyr_y", "gyr_z",
        "mag_x", "mag_y", "mag_z",
    ]

    for col in sensor_columns:
        x = window[col].to_numpy()

        features[f"{col}_mean"] = np.mean(x)
        features[f"{col}_std"] = np.std(x)
        features[f"{col}_var"] = np.var(x)
        features[f"{col}_min"] = np.min(x)
        features[f"{col}_max"] = np.max(x)
        features[f"{col}_median"] = np.median(x)
        features[f"{col}_range"] = np.max(x) - np.min(x)
        features[f"{col}_iqr"] = np.percentile(x, 75) - np.percentile(x, 25)
        features[f"{col}_mad"] = np.mean(np.abs(x - np.mean(x)))
        features[f"{col}_rms"] = np.sqrt(np.mean(x ** 2))
        features[f"{col}_energy"] = np.sum(x ** 2)

        if np.std(x) < 1e-8:
            features[f"{col}_skew"] = 0.0
            features[f"{col}_kurtosis"] = 0.0
        else:
            features[f"{col}_skew"] = skew(x)
            features[f"{col}_kurtosis"] = kurtosis(x)

        features[f"{col}_zero_cross"] = np.sum(np.diff(np.signbit(x)))

        freq, power = periodogram(x, fs=FS)
        features[f"{col}_spectral_energy"] = np.sum(power)
        if len(power):
            features[f"{col}_dominant_freq"] = freq[np.argmax(power)]
            p = power / (power.sum() + 1e-12)
            features[f"{col}_spectral_entropy"] = -np.sum(p * np.log2(p + 1e-12))

    features["body_sma"] = (
        np.abs(window["body_x"]).sum() + np.abs(window["body_y"]).sum() + np.abs(window["body_z"]).sum()
    ) / len(window)
    features["gyr_sma"] = (
        np.abs(window["gyr_x"]).sum() + np.abs(window["gyr_y"]).sum() + np.abs(window["gyr_z"]).sum()
    ) / len(window)
    features["mag_sma"] = (
        np.abs(window["mag_x"]).sum() + np.abs(window["mag_y"]).sum() + np.abs(window["mag_z"]).sum()
    ) / len(window)

    axis_groups = {
        "body": ["body_x", "body_y", "body_z"],
        "gyr": ["gyr_x", "gyr_y", "gyr_z"],
        "mag": ["mag_x", "mag_y", "mag_z"],
    }
    for prefix, axes in axis_groups.items():
        features[f"{prefix}_corr_xy"] = np.nan_to_num(window[axes[0]].corr(window[axes[1]]))
        features[f"{prefix}_corr_xz"] = np.nan_to_num(window[axes[0]].corr(window[axes[2]]))
        features[f"{prefix}_corr_yz"] = np.nan_to_num(window[axes[1]].corr(window[axes[2]]))

    return features

In [7]:
import time

acc_df = final_df[final_df["sensor"] == "acc"].copy()
gyr_df = final_df[final_df["sensor"] == "gyr"].copy()
mag_df = final_df[final_df["sensor"] == "mag"].copy()

print("Pre-indexing gyr and mag by (participant, activity, position)...")
t0 = time.time()

gyr_lookup = {
    key: g.sort_values("attr_time")
    for key, g in gyr_df.groupby(["participant", "activity", "position"], observed=True)
}
mag_lookup = {
    key: g.sort_values("attr_time")
    for key, g in mag_df.groupby(["participant", "activity", "position"], observed=True)
}

print(f"gyr_lookup: {len(gyr_lookup)} groups | mag_lookup: {len(mag_lookup)} groups")
print(f"Pre-indexing took {time.time() - t0:.1f} seconds\n")

all_features = []
n_segments_processed = 0
n_segments_skipped_short = 0
n_segments_skipped_missing = 0

group_cols = ["participant", "activity", "position", "segment_id"]
MIN_SEGMENT_LENGTH = WINDOW_SIZE

# --- progress tracking setup ---
TOTAL_SEGMENTS = final_df.groupby(group_cols, observed=True).ngroups
PRINT_EVERY = 500   # print a progress line every N segments
loop_start = time.time()

print(f"Total segments to iterate: {TOTAL_SEGMENTS}\n")

for i, ((participant, activity, position, segment_id), acc_segment) in enumerate(
    acc_df.groupby(group_cols, observed=True), start=1
):

    acc_segment = acc_segment.sort_values("attr_time").copy()

    if len(acc_segment) < MIN_SEGMENT_LENGTH:
        n_segments_skipped_short += 1
    else:
        t_min, t_max = acc_segment["attr_time"].min(), acc_segment["attr_time"].max()
        pad = MERGE_TOLERANCE_MS
        key = (participant, activity, position)
        gyr_full = gyr_lookup.get(key)
        mag_full = mag_lookup.get(key)

        if gyr_full is None or mag_full is None:
            n_segments_skipped_missing += 1
        else:
            gyr_segment = gyr_full[gyr_full["attr_time"].between(t_min - pad, t_max + pad)]
            mag_segment = mag_full[mag_full["attr_time"].between(t_min - pad, t_max + pad)]

            if len(gyr_segment) < MIN_SEGMENT_LENGTH or len(mag_segment) < MIN_SEGMENT_LENGTH:
                n_segments_skipped_missing += 1
            else:
                for axis in ["attr_x", "attr_y", "attr_z"]:
                    body, _ = gravity_filter(acc_segment[axis].to_numpy())
                    acc_segment[f"body_{axis[-1]}"] = body

                acc_processed = acc_segment[["attr_time", "body_x", "body_y", "body_z"]]
                gyr_processed = gyr_segment.rename(
                    columns={"attr_x": "gyr_x", "attr_y": "gyr_y", "attr_z": "gyr_z"}
                )[["attr_time", "gyr_x", "gyr_y", "gyr_z"]]
                mag_processed = mag_segment.rename(
                    columns={"attr_x": "mag_x", "attr_y": "mag_y", "attr_z": "mag_z"}
                )[["attr_time", "mag_x", "mag_y", "mag_z"]]

                merged = pd.merge_asof(
                    acc_processed, gyr_processed, on="attr_time",
                    direction="nearest", tolerance=MERGE_TOLERANCE_MS
                )
                merged = pd.merge_asof(
                    merged, mag_processed, on="attr_time",
                    direction="nearest", tolerance=MERGE_TOLERANCE_MS
                )
                merged = merged.dropna().reset_index(drop=True)

                if len(merged) < WINDOW_SIZE:
                    n_segments_skipped_short += 1
                else:
                    for window in create_sliding_windows(merged, WINDOW_SIZE, STEP_SIZE):
                        feature_vector = extract_features(window)
                        feature_vector["participant"] = participant
                        feature_vector["activity"] = activity
                        feature_vector["position"] = position
                        feature_vector["segment_id"] = segment_id
                        all_features.append(feature_vector)

                    n_segments_processed += 1

    # --- progress print ---
    if i % PRINT_EVERY == 0 or i == TOTAL_SEGMENTS:
        elapsed = time.time() - loop_start
        rate = i / elapsed
        remaining = (TOTAL_SEGMENTS - i) / rate if rate > 0 else float("inf")
        print(
            f"[{i}/{TOTAL_SEGMENTS}] "
            f"processed={n_segments_processed} "
            f"skipped_short={n_segments_skipped_short} "
            f"skipped_missing={n_segments_skipped_missing} | "
            f"elapsed={elapsed/60:.1f} min | "
            f"~{remaining/60:.1f} min remaining | "
            f"windows_so_far={len(all_features)}"
        )

print(f"\nDone in {(time.time() - loop_start)/60:.1f} minutes")
print(f"Segments processed          : {n_segments_processed}")
print(f"Segments skipped (too short): {n_segments_skipped_short}")
print(f"Segments skipped (missing sensor data): {n_segments_skipped_missing}")

features_df = pd.DataFrame(all_features)
print("Final features shape:", features_df.shape)

Pre-indexing gyr and mag by (participant, activity, position)...
gyr_lookup: 838 groups | mag_lookup: 838 groups
Pre-indexing took 12.4 seconds

Total segments to iterate: 18876



/home/reem2005/.local/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/reem2005/.local/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[500/18876] processed=467 skipped_short=30 skipped_missing=3 | elapsed=2.3 min | ~85.1 min remaining | windows_so_far=11742
[1000/18876] processed=929 skipped_short=64 skipped_missing=7 | elapsed=5.0 min | ~88.5 min remaining | windows_so_far=25694
[1500/18876] processed=1416 skipped_short=76 skipped_missing=8 | elapsed=7.5 min | ~87.2 min remaining | windows_so_far=39246
[2000/18876] processed=1892 skipped_short=100 skipped_missing=8 | elapsed=10.0 min | ~84.1 min remaining | windows_so_far=52318
[2500/18876] processed=2360 skipped_short=132 skipped_missing=8 | elapsed=12.2 min | ~79.9 min remaining | windows_so_far=63901
[3000/18876] processed=2828 skipped_short=163 skipped_missing=9 | elapsed=14.7 min | ~77.8 min remaining | windows_so_far=77277
[3500/18876] processed=3315 skipped_short=175 skipped_missing=10 | elapsed=17.1 min | ~75.2 min remaining | windows_so_far=89889
[4000/18876] processed=3795 skipped_short=192 skipped_missing=13 | elapsed=19.7 min | ~73.4 min remaining | wind

## Segment-Aware Synchronization + Feature Extraction

For every continuous segment (identified by `segment_id` within each
participant/activity/position group), we:
1. Apply the gravity filter to the accelerometer signal
2. Synchronize acc/gyr/mag via `merge_asof` (nearest match within tolerance)
3. Slide fixed-length windows across the segment and extract features

Because iteration happens per-segment, no window or filter operation
ever spans a recording gap.

Sanity Checks

In [8]:
print("Any missing values:", features_df.isna().sum().sum())
print("\nWindows per activity:")
print(features_df["activity"].value_counts())
print("\nWindows per participant:")
print(features_df["participant"].value_counts().sort_index())

features_df.head()

Any missing values: 0

Windows per activity:
activity
running         68882
lying           62692
sitting         62525
walking         62430
standing        61542
climbingup      59670
climbingdown    48295
jumping          9259
Name: count, dtype: int64

Windows per participant:
participant
proband1     28548
proband10    27340
proband11    28642
proband12    27257
proband13    28232
proband14    27576
proband15    28567
proband2     25608
proband3     29629
proband4     33200
proband5     31950
proband6     28113
proband7     29588
proband8     31779
proband9     29266
Name: count, dtype: int64


,body_x_mean,body_x_std,body_x_var,body_x_min,body_x_max,body_x_median,body_x_range,body_x_iqr,body_x_mad,body_x_rms,body_x_energy,body_x_skew,body_x_kurtosis,body_x_zero_cross,body_x_spectral_energy,body_x_dominant_freq,body_x_spectral_entropy,body_y_mean,body_y_std,body_y_var,body_y_min,body_y_max,body_y_median,body_y_range,body_y_iqr,...,mag_z_mad,mag_z_rms,mag_z_energy,mag_z_skew,mag_z_kurtosis,mag_z_zero_cross,mag_z_spectral_energy,mag_z_dominant_freq,mag_z_spectral_entropy,body_sma,gyr_sma,mag_sma,body_corr_xy,body_corr_xz,body_corr_yz,gyr_corr_xy,gyr_corr_xz,gyr_corr_yz,mag_corr_xy,mag_corr_xz,mag_corr_yz,participant,activity,position,segment_id
0,-0.013312,0.034942,0.001221,-0.099193,0.067895,-0.014541,0.167088,0.037801,0.026692,0.037392,0.139817,-0.041324,-0.101899,26,0.002442,0.5,4.289833,-0.001022,0.052340,0.002740,-0.196473,0.124799,-0.002177,0.321272,0.051679,...,0.000000,8.541000,7294.868100,0.000000,0.000000,0,0.000000,0.0,-0.000000,0.130143,0.065356,66.20488,-0.056975,0.043867,0.152290,-0.120569,0.453541,-0.613501,-1.145233e-14,0.000000e+00,0.000000e+00,proband1,climbingdown,chest,1
1,0.001922,0.043612,0.001902,-0.099193,0.118463,0.002581,0.217656,0.049927,0.034279,0.043654,0.190567,0.130511,-0.001179,15,0.003804,0.5,4.059204,0.001261,0.054277,0.002946,-0.196473,0.124799,-0.000502,0.321272,0.057962,...,0.000000,8.541000,7294.868100,0.000000,0.000000,0,0.000000,0.0,-0.000000,0.129010,0.065506,66.11609,-0.064799,-0.099952,0.202797,-0.119754,0.477623,-0.592597,-3.000504e-14,0.000000e+00,0.000000e+00,proband1,climbingdown,chest,1
2,0.006834,0.042727,0.001826,-0.077613,0.118463,0.005763,0.196076,0.064080,0.034446,0.043270,0.187232,0.303122,-0.336565,17,0.003651,1.0,4.272679,-0.002082,0.048478,0.002350,-0.158838,0.125513,-0.003773,0.284351,0.056050,...,0.078285,8.587750,7374.944445,1.960392,1.843137,0,0.024033,0.5,2.827348,0.126286,0.065775,66.21232,0.102075,-0.057787,0.081739,0.065038,0.371056,-0.463318,2.450713e-14,2.903576e-01,2.349659e-14,proband1,climbingdown,chest,1
3,-0.010086,0.038397,0.001474,-0.107254,0.106087,-0.016177,0.213341,0.046962,0.031332,0.039700,0.157606,0.356674,0.135641,28,0.002949,1.0,4.165225,0.005655,0.047445,0.002251,-0.146539,0.125513,0.006722,0.272052,0.044856,...,0.139685,8.741776,7641.865595,-0.628971,-1.604396,0,0.042883,0.5,1.707531,0.124160,0.068591,66.30111,0.187398,0.145190,0.011396,0.099018,0.405679,-0.434695,5.019650e-14,9.177195e-01,1.152457e-14,proband1,climbingdown,chest,1
4,-0.008177,0.035113,0.001233,-0.107254,0.088120,-0.007674,0.195375,0.043442,0.027607,0.036053,0.129979,0.037294,0.357301,29,0.002466,9.5,4.820722,0.004689,0.058465,0.003418,-0.216583,0.122389,0.008268,0.338971,0.056371,...,0.136435,8.745826,7648.946688,-0.675521,-1.543672,0,0.041476,0.5,1.724136,0.134652,0.068857,66.18664,0.025172,0.156597,0.219557,0.120205,0.456632,-0.304178,1.000000e+00,1.434992e-14,1.406210e-14,proband1,climbingdown,chest,1


In [9]:
features_df.to_pickle(FEATURES_OUTPUT)
print("Saved to:", FEATURES_OUTPUT)
print("File size (MB):", round(FEATURES_OUTPUT.stat().st_size / 1e6, 2))

Saved to: /lustre09/project/6081099/reem2005/DATASET/PROCESSED/realworld_features.pkl
File size (MB): 580.69
